# Towards Data: Linear Regression on the Prostate Data

This class, we consider the prostate cancer data, which is discussed in the classic [The elements of statistical learning](https://hastie.su.domains/ElemStatLearn/). This dataset, along with others treated in the book, can be found on [Hastie's webpage](https://hastie.su.domains/ElemStatLearn/data.html). The prostate cancer data itself is based on a paper by [Stamey (1989)](https://pubmed.ncbi.nlm.nih.gov/2468795/).

In [ ]:
from pathlib import Path
import csv

To get an idea of the data, we can read the accompanying `info.txt`

In [ ]:
info_path = Path("./prostate.info.txt")
with open(info_path, "r") as info_f:
    info = info_f.read()
print(info)

Note that this info file is Hastie's original. It speaks of a $96 \times 8$ feature matrix, but the data contain 97 samples (67 training and 30 test samples), as we will see below.

Next we read the data itself

In [ ]:
data_path = Path("./prostate.data")

Making use of `csv.reader` (or similar), implement a function `read_csv(path)` that 
* takes a string or `Path` object as input
* checks that the file exists at the given path (and raises an error otherwise)
* returns as second argument the file's first row as `col_names`, without its first entry: the header has no name for the first column, which holds a running index, so this entry is an empty string
* and the remaining rows in the data as its first argument `data`

Hint: CSV stands for "comma separated values" but when you inspect `prostate.data`, you'll see no commas.

Hint 2: The csv reader takes a `delimiter` argument.

In [ ]:
def read_csv(path: str | Path) -> tuple[list, list]:
    """Return the csv data located at path as a list of lists and the header as list of column names."""
    path = Path(path)  # also accept paths
    if not path.exists():
        raise FileNotFoundError(f"Data not found at {path}")
    with open(path, "r") as f:
        reader = csv.reader(f, delimiter='\t')
        col_names = next(reader)  # get column names
        data = [row for row in reader]

    assert col_names.pop(0) == '', "To be removed column name is not empty string"
    return data, col_names

Having done so, take a look at the data. Specifically, what is the type of the entries?

In [ ]:
data_raw, names = read_csv(data_path)
for entry in data_raw[0]:
    print(entry, type(entry).__name__, sep=': ')

In [ ]:
data_raw[0]

All of the values are saved as strings. This is a rather common problem you may face in practice so it's worth knowing how to deal with it. 

Implement a function `try_num` which tries to convert a string input to the correct numeric value (integer or float) but returns the value as is, in case that does not work.

Should you raise an exception, in case the input is not a string?

In [ ]:
int("9.99")
# float("9.99")

In [ ]:
def try_num(value: str):
    """Try to convert a numeric value, which is given as a string, to a numeric value."""
    # if not isinstance(value, str): raise TypeError(f"expected string but input is of type {type(value)}")
    try:
        return int(value)
    except ValueError:
        try:
            return float(value)
        except ValueError:
            return value

In [ ]:
for val in ['5', '10.0', 'proper string', 5]:
    print(val, try_num(val), sep=": ")

In [ ]:
# if data_raw[0][-1] == "T":
#     data_raw[0][-1] = ...
data_raw

**Answer:** Both choices can be argued for. Here, `try_num` is meant for strings that we read from a file, so a non-string input points to a bug elsewhere in our code. Raising a `TypeError` makes such a bug visible immediately, instead of letting it pass silently, as the last loop iteration above shows. If we expected lists that mix strings and numbers, returning non-strings unchanged would be the more convenient choice.

Next we use `try_num` in a `preprocess_data` function. Write a function that takes a list of lists as input and converts its string components to numeric values if possible. While you're at it, you may have noticed that the first entry of the lists is a running index and the last entry is a string that indicates whether the sample belongs to the training set or not. Remove the running index and convert the training flag to a bool, `True` for `'T'` and `False` for `'F'`.

Return, again as a list of lists, the preprocessed data.

In [ ]:
def preprocess_data(data: list[list]) -> list[list]:
    """Given a list of lists, convert strings to numeric values if possible, remove the running index and replace the last entry,
    indicating whether the sample belongs to the train set, with a bool."""
    data_preproc = [[try_num(val) for val in row] for row in data]
    for idx, row in enumerate(data_preproc, 1): # start enumerate from 1
        row.remove(idx)
        row[-1] = (row[-1] == "T")  # True or False
    return data_preproc

In [ ]:
data = preprocess_data(data_raw)
for entry in data:
    for element in entry:
        assert isinstance(element, (int, float, bool))  # asserting that elements are either int, float or a boolean

Most, if not all datasets, in the realm of machine learning are composed of a train/val/test split.

Implement a function `split_train_val_test` that takes as input a list of lists, and splits them into 3 chunks. A train, a validation and a test split. The train split shall consist of all samples that read `True` in their boolean Train entry (should be the last entry in the sample). The remaining chunk will further, after random shuffling, be split into validation and test set each of equal size. 



In [ ]:
from random import shuffle

def split_train_val_test(data: list, is_train_idx: int = -1):
    """Splits a list of lists into train, val and test set, assuming that the last feature dimension designates train samples."""
    ### BEGIN SOLUTION
    # STUB: raise NotImplementedError()  # remove after having implemented your solution
    train = [sample for sample in data if sample[is_train_idx]]

    not_train = [sample for sample in data if not sample[is_train_idx]]
    shuffle(not_train) # shuffle list (in place) to not induce a systematic bias
    val = not_train[:len(not_train) // 2]
    test = not_train[len(not_train) // 2:]

    return train, val, test
    ### END SOLUTION


def split_xy(data: list, x_start: int, x_end: int, y_idx: int) -> tuple[list, list]:
    """Splits a list of lists into predictors (indicated by x_start and x_end) and labels (indicated by y_idx).

    Note, that we assume a single y value."""
    x = [sample[x_start : x_end] for sample in data]  # x_end is exclusive
    y = [sample[y_idx] for sample in data]

    return x, y

In [ ]:
from random import seed

seed(0)  # fix the random shuffle, so that we all get the same split
train, val, test = split_train_val_test(data)
assert len(val) == len(test)

Next up we split the data into predictors `x` and targets `y`. 

Implement a function `split_xy` that takes as input a list of lists and returns a `x, y`, which is a tuple of lists. 

As we have specific data at hand you can hardcode the index ranges or, if you're up to it, pass the index ranges of x and y entries as an argument to the function

In [ ]:
x_train, y_train = split_xy(train, 0, 8, 8)
x_val, y_val = split_xy(val, 0, 8, 8)
x_test, y_test = split_xy(test, 0, 8, 8)

In [ ]:
for sample in x_train:
    assert len(sample) == 8

We can think of the lists of lists that we are dealing with as $N \times D$ matrices where $N$ is the number of samples, and $D$ the feature dimension, i.e. the number of values that describe every sample.

In [ ]:
print(f"Number of features: D = {len(x_train[0])}")
print(f"Number of training samples: N_train = {len(x_train)}")
print(f"Number of validation samples: N_val = {len(x_val)}")
print(f"Number of test samples: N_test = {len(x_test)}")

Our goal is to learn a model from the training set that predicts, given a sample's features $\mathbf{x}_i \in \mathbb{R}^D$, its label $y_i \in \mathbb{R}$. We want to use the validation set to determine how our model performs on the unseen data.

The test set should not be made use of until we are content with the performance on the validation set. Certainly, the test set should **not** be used to finetune the model, as it serves as a proxy of how well our model *generalizes* to unseen data.

### Linear regression: A model *linear in weights* to predict real values. 

In the general set up of linear regression, we assume that we can estimate $i$'s sample's label $\hat{y}_i$ by appropriately combining its features $\mathbf{x}_i$. This combination is weighted by $\mathbf{w}$ and corrected for a general offset by $b$,

$$ \hat{y}_i = \mathbf{w}^\top \mathbf{x}_i  + b.$$

Now the error, also called *residual*, $\epsilon$ is the deviation between the ground truth target $y$ and our estimated one $\hat{y}$,

$$ \epsilon_i := y_i -\hat{y}_i  \quad \implies \quad y_i = \mathbf{w}^\top \mathbf{x}_i  + b + \epsilon_i.$$

Naturally, we want our model to minimize this error, not only for a specific sample but for all samples at once. Simply summing the residuals does not work, as positive and negative residuals would cancel out. Hence, we sum the *squared* residuals, which are never negative and pay more attention to large errors than to small ones (besides other nice mathematical properties). Formally, we define our *Loss* $L$ as the sum of squared residuals,
also known as the *(Ordinary) Least Squares* objective:

$$ L = \sum_i^N \epsilon_i^2 = \sum_i^N (y_i - \mathbf{w}^\top \mathbf{x}_i - b)^2 .$$

We want to find parameters $\mathbf{w}$ and $b$ that minimize it,

$$ (\hat{\mathbf{w}}, \hat{b}) =  \underset{\mathbf{w},\, b}{\mathrm{arg\, min}} \ L.$$

In this set up, one may see a scaling of the loss by $\frac{1}{2}$ or $\frac{1}{N}$ to make computations or their interpretations simpler. However, this does not alter the result.

To make life easy, we absorb the bias $b$ in the weights and simultaneously extend all samples with a constant entry $1$, i.e. 

$$ \mathbf{w} \mapsto (b, w_1, ..., w_D)^\top \quad \text{and} \quad \mathbf{x}_i \mapsto (1, x_{i1}, ..., x_{iD})^\top \ \text{for} \ i=1, ..., N. $$

If this does not appear reasonable to you, verify quickly by pen and paper that the expression for $\hat{y}_i$ remains the same. 

<br>


This simplifies the objective to only solving for $\mathbf{w}$

$$ \hat{\mathbf{w}} = \underset{\mathbf{w}}{\mathrm{arg\, min}} \sum_i^N (y_i - \mathbf{w}^\top \mathbf{x}_i)^2 .$$

Writing the inner product of weights and sample out we have 

$$ \hat{\mathbf{w}} = \underset{\mathbf{w}}{\mathrm{arg\, min}} \sum_i^N  (y_i - \sum_j^{D+1} w_j x_{ij})^2.$$

Setting every partial derivative $\frac{\partial}{\partial w_k} \ L$ to zero gives $D+1$ equations. These cannot be solved one weight at a time, as every equation contains all weights: they form a coupled system of linear equations. In matrix notation, we can solve this system for the whole vector $\hat{\mathbf{w}}$ at once by setting $\frac{\partial}{\partial \mathbf{w}} \ L \overset{!}{=} 0$, which yields $\mathbf{X}^\top \mathbf{X} \hat{\mathbf{w}} = \mathbf{X}^\top \mathbf{y}$ and thus

$$ \hat{\mathbf{w}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

where $\mathbf{X}$ is the $N \times D (+1)$ dimensional matrix with samples as rows and $\mathbf{y}$ is the $N$ dimensional column vector of (known) sample targets. The system $\mathbf{X}^\top \mathbf{X} \hat{\mathbf{w}} = \mathbf{X}^\top \mathbf{y}$ is known as the *Normal Equations*.

### Question: Why is the point where the derivative of the loss with respect to the weights vanishes a minimum (and not a maximum or a saddle point)?

**Answer:** The loss $L(\mathbf{w}) = \lVert \mathbf{y} - \mathbf{X}\mathbf{w} \rVert^2$ is a quadratic function of $\mathbf{w}$ with second derivative (Hessian) $2\mathbf{X}^\top \mathbf{X}$. This matrix is positive semidefinite, since $\mathbf{v}^\top \mathbf{X}^\top \mathbf{X} \mathbf{v} = \lVert \mathbf{X}\mathbf{v} \rVert^2 \geq 0$ for every vector $\mathbf{v}$. Hence, $L$ is *convex*, i.e. bowl-shaped, and for a convex function every point with vanishing derivative is a global minimum. If the columns of $\mathbf{X}$ are linearly independent, $\mathbf{X}^\top \mathbf{X}$ is even positive definite and thus invertible, so the minimum is unique. That is exactly what we need to compute $(\mathbf{X}^\top \mathbf{X})^{-1}$ in the normal equations.

After this short summary of linear regression, we realize one thing: We need an implementation of linear algebra, specifically: matrix multiplication, transposition and inversion.


Let's start with multiplication.
First, implement a helper function `unsqueeze` that adds a dimension to a list, if the list is not already a list of lists. Specifically we need this function to deal with vectors such as e.g. `y`, wrapping every of their numeric values in a list.

Next, implement `matmul` which takes an input two lists (of lists) and multiplies the first from the left to the other. `matmul` should unsqueeze its inputs to make sure that the operations are valid.

Hint: `zip` takes as input a collection of iterables and returns collections of first all the iterables' first element, then all second elements and so on. This can be made use of to access the columns of the matrix.

In [ ]:
def unsqueeze(container: list) -> list[list]:
    """Adds a dimension to a list which might be useful for linear algebra operations.

    Note, that a dimension is only added when list entries are not already lists."""
    if all(not isinstance(element, list) for element in container):
        container = [[element] for element in container]
    return container

def matmul(x: list[list], y: list[list]) -> list[list]:
    """Simple implementation of matrix multiplication between x (from left) and y."""
    x = unsqueeze(x)
    y = unsqueeze(y)
    n_x = len(x)
    m_x = len(x[0])

    n_y = len(y)
    if not isinstance(y[0], (list, tuple)):
        y = [[elem for elem in y]]
    m_y = len(y[0])

    # Sanity checks
    assert m_x == n_y, f"Number of columns in x must match number of rows in y but, {m_x} != {n_y} "
    assert all(len(row_x) == m_x for row_x in x), "All rows in x must have the same length"
    assert all(len(row_y) == m_y for row_y in y), "All rows in y must have the same length"

    out = [
            [sum([element_x * element_y for element_x, element_y in zip(row_x, col_y)]) for col_y in zip(*y)]
            for row_x in x
            ]

    return out

Check your implementation here:

In [ ]:
t1 = [
    [0, 1, 2],
    [3, 4, 5]
]
t2 = [
    [0, 1],
    [2, 3],
    [4, 5]
]

id_2 = [
    [1, 0],
    [0, 1]
]

id_3 = [
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
]

assert matmul(id_2, t1) == t1
assert matmul(id_3, t2) == t2
assert matmul(t1, t2) == [[10, 13], [28, 40]]

Implement a little helper `transpose` to transpose a matrix (which you may have already used).

When you use `zip`, note that it does *not* return a `list`.


In [ ]:
def transpose(x: list[list]) -> list[list]:
    """Small wrapper to transpose a matrix"""
    return list(map(list, zip(*x)))

assert matmul(transpose(t2), transpose(t1)) == transpose(matmul(t1, t2))

We spare ourselves the implementation of matrix inversion in pure python and make use of [this stackoverflow answer](https://stackoverflow.com/questions/32114054/matrix-inversion-without-numpy).

In [ ]:
def eliminate(r1, r2, col, target=0):
    fac = (r2[col]-target) / r1[col]
    for i in range(len(r2)):
        r2[i] -= fac * r1[i]

def gauss(a: list[list]):
    for i in range(len(a)):
        if a[i][i] == 0:
            for j in range(i+1, len(a)):
                if a[j][i] != 0:  # look for a nonzero entry below the zero pivot
                    a[i], a[j] = a[j], a[i]
                    break
            else:
                raise ValueError("Matrix is not invertible")
        for j in range(i+1, len(a)):
            eliminate(a[i], a[j], i)
    for i in range(len(a)-1, -1, -1):
        for j in range(i-1, -1, -1):
            eliminate(a[i], a[j], i)
    for i in range(len(a)):
        eliminate(a[i], a[i], i, target=1)
    return a

def inverse(a: list[list]):
    tmp = [[] for _ in a]
    for i,row in enumerate(a):
        assert len(row) == len(a)
        tmp[i].extend(row + [0]*i + [1] + [0]*(len(a)-i-1))
    gauss(tmp)
    return [tmp[i][len(tmp[i])//2:] for i in range(len(tmp))]

To absorb the bias into the weights, we still have to prepend the features by a constant 1. Hence, implement `prepend_by_1` that prepends all list entries in a given list with 1.0.

In [ ]:
def prepend_by_1(x: list[list]) -> list[list]:
    """Prepend every list in a list of lists `x` with 1.0."""
    x = unsqueeze(x)
    return [[1.0] + sub_list for sub_list in x]

Now we have everything we need to solve for the weights. Write a function `get_ssq_weights` that takes as input a list of list `x` and a list (not necessarily of lists) `y` and returns `w` which is defined by 

$$ \hat{\mathbf{w}} = (X^\top X)^{-1}X^\top \mathbf{y} $$

In [ ]:
def get_ssq_weights(x: list[list], y: list) -> list:
    """Solves for the weights `w` that minimize the sum of squared residuals of the linear regression problem

    y = w^T x + epsilon."""
    x = prepend_by_1(x)  # x -> (1, x)
    y = unsqueeze(y)

    xt = transpose(x)
    xt_x = matmul(xt, x)
    xt_x_inv = inverse(xt_x)

    xt_x_inv_xt = matmul(xt_x_inv, xt)
    w =  matmul(xt_x_inv_xt, y)  # (b, w)
    return w


Now obtain `w` and make your prediction on the validation set, that is compute

$$ \hat{y}_{\text{val}} = X_{\text{val}} \mathbf{w} $$

where $X_{\text{val}}$ is again prepended by a column of ones.

In [ ]:
w = get_ssq_weights(x_train, y_train)
y_hat = matmul(prepend_by_1(x_val), w)


Next up, we want to have some measure for how good our fit is. More sophisticated choices exist, but we'll go for a simple average sum of squared residuals.

First, implement a helper `squeeze` that does the opposite of `unsqueeze` and takes a list of lists to a list if the entries of the lists are all lists of length one.

Then, implement a function `get_ssq_avg` that takes two lists (possibly of lists), your prediction `pred` and the corresponding `target`s, and computes the average squared residual.

In [ ]:
def squeeze(container: list[list]) -> list:
    """Flatten redundant dimensions in a list.

    If the list consists of lists which itself only consist of one element, squeeze returns a list of these elements."""
    # STUB: raise NotImplementedError()  # remove after having implemented your solution
    if all(isinstance(element, list) for element in container):
        if all(len(element) == 1 for element in container):
            container = [el_of_element for element in container for el_of_element in element]

    return container

def get_ssq_avg(pred: list, target: list) -> float:
    """Get the average squared residual for a lists of predictions and targets."""
    # STUB: raise NotImplementedError()  # remove after having implemented your solution
    n = len(pred)
    assert n == len(target), f"Number of predictions must match number of targets, but {n} != {len(target)}"
    pred = squeeze(pred)
    target = squeeze(target)

    return sum((pred_i - y_i)**2 for pred_i, y_i in zip(pred, target)) / n

In [ ]:
get_ssq_avg(y_hat, y_val)

But this value does not tell us much on its own. As a simple relative measure, we can compare it to the same error on the train set. Using the same weights `w` predict $\hat{\mathbf{y}}_{\text{train}}$ and compare it to the ground truth $\mathbf{y}_{\text{train}}$

In [ ]:
y_hat_train = matmul(prepend_by_1(x_train), w)
get_ssq_avg(y_hat_train, y_train)

Hopefully, this is smaller. At this point we may realize that we have forgotten something that may be of value to improve our result. We did not yet standardize the data.

### Why is standardization useful?

**Answer:** The features live on very different scales: `age` is around 40 to 80, `svi` is 0 or 1 and `pgg45` ranges from 0 to 100. After standardization, all features have mean 0 and standard deviation 1. This
* makes the weights comparable: every weight then states how much the prediction changes if its feature changes by one standard deviation,
* improves the numerical stability of computations such as inverting $\mathbf{X}^\top \mathbf{X}$,
* matters for all methods that penalize large weights (e.g. ridge regression), that compare distances between samples (e.g. $k$ nearest neighbors) or that are trained by gradient descent.

For ordinary least squares, however, standardization does not change the predictions, as we will see in a moment.


### Exercise: Implement
* `get_mean_feature_vec` that takes a list of lists as input and returns a list of feature length, i.e. a mean array
* `get_feature_stds` that takes a list of lists as input and returns a list of feature length, i.e. a standard deviation array
* `get_standardized` that takes a list of lists as input and returns the same shape list of lists, standardized, i.e. computes for every sample component 

$$x_{ij} \leftarrow \frac{x_{ij} - \mu_j}{\sigma_j}$$

where $\mu_j$ and $\sigma_j$ are the $j$'th components of the mean and standard deviation arrays. Additionally, `get_standardized` should return the mean and standard deviation arrays. Optionally it should also accept an already computed mean and standard deviation. If these are not given, they shall be computed on the fly.

In [ ]:
def get_mean_feature_vec(x: list[list]) -> list:
    """Computes the sample mean of a list of samples."""
    x = unsqueeze(x)
    n = len(x)
    x_t = transpose(x)
    return [sum(feat_col)/n for feat_col in x_t]

def get_feature_stds(x: list[list], mean: list = None):
    """Compute the standard deviation for data `x` given as list of lists.

    If the mean vector is not given it will be computed on the fly."""
    mean = get_mean_feature_vec(x) if mean is None else mean

    x_t = transpose(x)

    n = len(x)
    stds = []
    for feature_j, mean_j in zip(x_t, mean):
        sum_j = 0.0
        for feature_ij in feature_j:
            sum_j += (feature_ij - mean_j)**2
        stds.append((sum_j / (n-1))**(1/2))

    return stds

def get_standardized(x: list[list], mean: list = None, stds: list = None):
    """Standardize data `x` by subtracting feature means and dividing by standard deviations."""
    mean = get_mean_feature_vec(x) if mean is None else mean
    stds = get_feature_stds(x) if stds is None else stds

    x_s = []  # matrix of standardized values
    for x_i in x:
        x_i_s = []  # row
        for x_ij, mean_j, std_j in zip(x_i, mean, stds):
            x_i_s.append((x_ij - mean_j) / std_j)
        x_s.append(x_i_s)

    return x_s, mean, stds

In [ ]:
x_train_stand, mu_train, sigma_train = get_standardized(x_train)

If everything worked out nicely, the standardized data should have roughly mean 0 and variance 1.

In [ ]:
mu_stand = get_mean_feature_vec(x_train_stand)
sigma_stand = get_feature_stds(x_train_stand)

assert all(abs(m) < 1e-10 for m in mu_stand)
assert all(abs(s - 1.0) < 1e-4 for s in sigma_stand)

Now we want to predict after training on the standardized data again. For that we have to transform the validation set with the mean and standard deviation that we obtained from the *training* set. That was the reason for returning these statistics in the first place. To use these parameters and not the validation set's own ones is important to (1) avoid *data leakage* and (2) remain consistent. Data leakage refers to an unfair advantage we would obtain by knowing the distribution of the set we want to make estimations of. Consistent here means that we trained the model to predict on values that have been standardized by the statistics of the training data, and this assumption will be kept on distributions outside of the training data.

Now predict new estimates on the validation set, after transforming it with mean and standard deviation of the train set.

In [ ]:
x_val_stand, _, _ = get_standardized(x_val, mean = mu_train, stds = sigma_train)
w_stand = get_ssq_weights(x_train_stand, y_train)
y_hat_stand = matmul(prepend_by_1(x_val_stand), w_stand)

In [ ]:
x_val[0], x_val_stand[0]

Note, that we do not standardize the predicted variables $y$. Otherwise, we would have to "destandardize" them.

In [ ]:
get_ssq_avg(y_hat_stand, y_hat)
# get_ssq_avg(y_hat_stand, y_val)

Wait, what happened here. Why are the results the same?

**Answer:** Standardization replaces every feature by an affine function of itself, $x_{ij} \mapsto (x_{ij} - \mu_j)/\sigma_j$. A linear model with intercept can undo this exactly: with weights $w_j \sigma_j$ and bias $b + \sum_j \mu_j w_j$, the model on standardized data makes the same predictions as the model with weights $w_j$ and bias $b$ on raw data. Hence, both models can represent exactly the same set of functions, and least squares finds the same best function in both cases. Only the weights differ, the predictions agree (up to rounding errors of order $10^{-21}$). This changes as soon as we penalize the size of the weights, as in ridge regression on sheet 2.

Now we do the same thing with `numpy`. We will worry about the details of numpy later but the quick dive will do for now:

You can simply wrap a list, or lists of lists, etc., with `np.array` to make a numpy array out of it.

#### Create numpy arrays out of the unstandardized data lists (`x_train`, `y_train`, `x_val`...)

In [ ]:
import numpy as np

x_train_np = np.array(x_train)
y_train_np = np.array(y_train)

x_val_np = np.array(x_val)
y_val_np = np.array(y_val)

x_test_np = np.array(x_test)
y_test_np = np.array(y_test)

#### Standardize the numpy data. 

You can assume that addition and subtraction works for numpy arrays as you would expect for vectors, matrices, etc. - i.e. elementwise.

Furthermore, you'll witness one neat aspect of numpy arrays: *Broadcasting*.

In short and specifically in our case, you can subtract a vector `vec` of length `D` from every row of a `NxD` matrix `mat` by just doing `mat - vec`. 

This also works for multiplication/division. If the factor `fac` to multiply with is a single number, multiplication `*` with an array `arr` of whatever dimension will occur for every single element via `fac * arr`.

You'll further need the `mean` and standard deviation `std` methods of numpy arrays. This is where, for multidimensional arrays, another numpy concept kicks in - that of `axis`. You can think of `axis` as of a dimension. For 2D arrays, the first dimension (`axis=0`) indexes the rows, while the second dimension (`axis=1`) does so for the columns. We want to compute the mean and standard deviation of our features across samples. Every sample occupies one row, so you have to compute these values along `axis=0`. 

Furthermore, note that we computed the mean from the data itself such that the normalization of the standard deviation has to scale with (the square root) of `1/(n-1)` rather than the default `1/n`. Numpy's `std` method takes a `ddof` argument that you can set to the delta of degrees of freedom.


In [ ]:
def get_standardized_np(x: np.ndarray, mean: np.ndarray = None, stds: np.ndarray = None):
    """Standardize data `x` by subtracting feature means and dividing by standard deviations."""
    mean = x.mean(axis=0) if mean is None else mean
    stds = x.std(axis=0, ddof=1) if stds is None else stds

    x_stand = (x - mean) / stds

    return x_stand, mean, stds

x_train_np_stand, mu_train_np, sigma_train_np = get_standardized_np(x_train_np)

If you've done so correctly, the mean of your standardized data should be (very close to) 0, and the standard deviations (very close to) 1.
Verify for the training data:

In [ ]:
want_mean = np.zeros(x_train_np.shape[1])
want_stds = np.ones(x_train_np.shape[1])

# assign mean and std of standardized data
is_mean = x_train_np_stand.mean(axis=0)
is_std = x_train_np_stand.std(axis=0, ddof=1)

np.testing.assert_almost_equal(want_mean, is_mean)
np.testing.assert_almost_equal(want_stds, is_std, decimal=2)

Next, we want to obtain the weights that minimize the least squares objective. For this, make use of
* `np.linalg.inv(mat)` to obtain the matrix inverse of `mat`,
*  the `mat.T` attribute to obtain the transpose of `mat` and
*  the `@` operator to multiply matrices with each other (or vectors) 

Recall also, that in order to obtain the bias $b$, we have to prepend all the data samples with 1. 
Make use of [`np.insert`](https://numpy.org/doc/stable/reference/generated/numpy.insert.html) to achieve this for numpy array data.

In [ ]:
def prepend_by_1_np(x: np.ndarray, axis: int=1) -> np.ndarray:
    """Small wrapper to prepend a numpy array with 1.0 along a certain axis."""
    # STUB: raise NotImplementedError()  # remove after having implemented your solution
    return np.insert(x, 0, 1.0, axis=axis)


def get_ssq_weights_np(x: np.ndarray, y: np.ndarray, fit_intercept: bool = True) -> np.ndarray:
    """Computes the weights `w` that solve an ordinary least squares problem in numpy.

    w = (x^T x)^-1 x^T y"""
    x = prepend_by_1_np(x) if fit_intercept else x
    w = np.linalg.inv(x.T @ x) @ x.T @ y
    return w

Assert that the weights you obtain via numpy match the ones we implemented in pure python. Likewise your prediction should also match. You might have to adjust shapes via `squeeze`.

Note that we want to work with the standardized data.

In [ ]:
w_np = get_ssq_weights_np(x_train_np_stand, y_train_np)

np.testing.assert_allclose(w_np, squeeze(w_stand))

x_val_np_stand, _, _ = get_standardized_np(x_val_np, mean=mu_train_np, stds=sigma_train_np)
y_hat_np_stand = prepend_by_1_np(x_val_np_stand) @ w_np

np.testing.assert_allclose(y_hat_np_stand, squeeze(y_hat_stand))

If you've come this far, congratulations. You've implemented linear algebra operations in pure python that match numpy, the standard library for numerical computing in Python. Linear algebra forms the basis for most implementations of Artificial Intelligence, so be sure to appreciate your effort.

With that you can well appreciate what designated libraries do under the hood. We will implement the same procedure in 1 to 2 lines of code via `sklearn`.

Use sklearn's `LinearRegression`'s `fit(x,y)` method to fit the model on our (standardized) training data.

You can work with both, our pure python list of lists implementation or numpy.

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression().fit(x_train_stand, y_train)


Assert that 
* the learned models `intercept_` attribute corresponds to our learned bias $b$
* the learned models `coef_` attribute corresponds to our learned weights $w$

In [ ]:
np.testing.assert_allclose(reg.intercept_, squeeze(w_stand)[0])  # check bias
np.testing.assert_allclose(reg.coef_, squeeze(w_stand)[1:])  # check weights

Next, use the model (output of the fit method) to predict ys on the validation x data and assert that it matches our predictions from before.

In [ ]:
y_hat_sk = reg.predict(x_val_stand)
np.testing.assert_allclose(y_hat_sk, squeeze(y_hat_stand))